# Kaggle Student Performance Factors Dataset
## Machine Learning Benchmark & Exploratory Data Analysis (EDA)

This notebook documents the end-to-end ML workflow on the real Kaggle **Student Performance Factors** dataset (6,607 records, 20 attributes).
- Dataset Inspection & Null Handling
- Categorical Encoding & Numerical Scaling
- Exploratory Data Analysis & Feature Correlations
- Benchmark Training: Linear Regression, Decision Tree, Random Forest, Gradient Boosting
- Model Evaluation Leaderboard (MAE, MSE, RMSE, R²)
- Model Selection & Artifact Serialization


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')


### 1. Load and Inspect Kaggle Dataset

In [ ]:
data_path = os.path.join('..', 'data', 'StudentPerformanceFactors.csv')
df = pd.read_csv(data_path)
print(f'Dataset Shape: {df.shape}')
display(df.head())
print('\nMissing Values Count:')
print(df.isnull().sum()[df.isnull().sum() > 0])

### 2. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['Exam_Score'], kde=True, color='#2563eb', bins=30)
plt.title('Distribution of Target Exam Score (Kaggle Dataset)', fontsize=14, fontweight='bold')
plt.xlabel('Exam Score')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.scatterplot(data=df, x='Hours_Studied', y='Exam_Score', ax=axes[0], alpha=0.5, color='#1e3a8a')
axes[0].set_title('Hours Studied vs Exam Score')

sns.scatterplot(data=df, x='Attendance', y='Exam_Score', ax=axes[1], alpha=0.5, color='#059669')
axes[1].set_title('Attendance % vs Exam Score')

sns.scatterplot(data=df, x='Previous_Scores', y='Exam_Score', ax=axes[2], alpha=0.5, color='#d97706')
axes[2].set_title('Previous Scores vs Exam Score')

plt.tight_layout()
plt.show()

### 3. Preprocessing & Encoding Pipeline

In [ ]:
import sys
sys.path.append(os.path.join('..', 'src'))
from preprocessing import KaggleStudentPreprocessor

preprocessor = KaggleStudentPreprocessor()
X_train_scaled, X_test_scaled, y_train, y_test, X_train_df, X_test_df = preprocessor.fit_transform_train(df)
print(f'Train Samples: {X_train_scaled.shape[0]}, Test Samples: {X_test_scaled.shape[0]}')
print(f'Number of Features: {X_train_scaled.shape[1]}')

### 4. Model Benchmarking & Evaluation Leaderboard

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=6, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=150, max_depth=10, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=150, learning_rate=0.08, max_depth=5, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)
    results.append({'Model': name, 'MAE': round(mae, 4), 'RMSE': round(rmse, 4), 'R2 Score': round(r2, 4)})

leaderboard = pd.DataFrame(results).sort_values(by='R2 Score', ascending=False)
display(leaderboard)